# 04. Dựng đồ thị chính thức (task 1.5, 1.6), cấu hình chia tập từ notebook 08

| Bộ | W = S | Chia | Số khối | Theo |
|---|---|---|---|---|
| NF-ToN-IoT-v3 (chính) | 5 s | blocked, gap 1 | 10 | số node |
| NF-CSE-CIC-IDS2018-v3 (thứ 3) | 5 s | blocked, gap 1 | 20 | số node |
| NF-ToN-IoT-v2 (phụ) | 1000 flow | blocked, gap 1 | 200 | số node |

Mỗi bộ ghi `graphs_*.pt`, `split.json`, `scaler.pkl`, `graph_stats.csv`, `summary.json` (gồm kết quả kiểm tra rò rỉ, NaN, gap, tái lập hash) vào `/kaggle/working/processed/<bộ>/`. Output của kernel này làm đầu vào cho các notebook sau qua `kernel_sources`.

In [ ]:
import glob, os, subprocess, sys, time

def run(cmd, ok=(0,), **kw):
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    print(r.stdout[-7000:], end="")
    print(r.stderr[-2500:], end="", file=sys.stderr)
    print(f"[mã thoát {r.returncode}]")
    if r.returncode not in ok:
        raise RuntimeError(f"Lệnh lỗi: {' '.join(cmd)}")
    return r.returncode

REPO = "/tmp/gnn-iot-ids"          # ngoài /kaggle/working để không lẫn vào output
run(["rm", "-rf", REPO])
run(["git", "clone", "--depth", "1", "https://github.com/minucedev/gnn-iot-ids.git", REPO])
run(["git", "-C", REPO, "log", "-1", "--format=%H %s"])
with open(f"{REPO}/requirements.txt") as f:
    reqs = [l.strip() for l in f if l.strip() and not l.startswith("#")]
keep = [r for r in reqs if r.split("==")[0].lower() not in {"torch", "jupyter", "ipykernel"}]
run([sys.executable, "-m", "pip", "install", "-q", *keep])

In [ ]:
OUT = "/kaggle/working/processed"
JOBS = [  # (bộ, file, W, S, số khối)
    ("nf_ton_iot_v3", "NF-ToN-IoT-v3.csv", 5, 5, 10),
    ("nf_cicids2018_v3", "NF-CICIDS2018-v3.csv", 5, 5, 20),
    ("nf_ton_iot_v2", "NF-ToN-IoT-v2.csv", 1000, 1000, 200),
]
codes = {}
for ds, fn, w, s, blocks in JOBS:
    csv = next(glob.iglob(f"/kaggle/input/**/{fn}", recursive=True))
    t0 = time.time()
    print("=====", ds, csv)
    # mã 3 = có kiểm tra không đạt: vẫn giữ output để xem summary.json, báo ở cuối
    codes[ds] = run([sys.executable, "-I", "scripts/build_dataset_cli.py", ds, csv, str(w), str(s), "blocked", "1",
                     f"{OUT}/{ds}", f"blocks={blocks}", "balance=1", f"tag={ds}_w{w}s{s}_blocked"],
                    ok=(0, 3), cwd=REPO)
    print(f"{ds} xong sau {time.time() - t0:.0f} s")
run(["du", "-sh", OUT])
print("Mã thoát từng bộ (0 = mọi kiểm tra đạt, 3 = có kiểm tra không đạt):", codes)